# Calendar Table

Creates `adb_qnxthealth.silver.calendar` — a standard date dimension table spanning 1940-01-01 to 2030-12-31 with date keys, year/month/quarter/week breakdowns, day names, weekend flags, and period boundary dates.

In [0]:
CREATE OR REPLACE TABLE adb_qnxthealth.silver.calendar AS
WITH date_range AS (
  SELECT explode(sequence(DATE '1940-01-01', DATE '2030-12-31', INTERVAL 1 DAY)) AS date
)
SELECT
  -- Key & core date
  CAST(date_format(date, 'yyyyMMdd') AS INT)       AS date_key,
  date                                              AS date,

  -- Year
  year(date)                                        AS year,
  CASE WHEN (year(date) % 4 = 0 AND year(date) % 100 != 0) OR (year(date) % 400 = 0)
       THEN TRUE ELSE FALSE END                     AS is_leap_year,

  -- Quarter
  quarter(date)                                     AS quarter,
  CONCAT('Q', quarter(date))                        AS quarter_name,
  CONCAT(year(date), '-Q', quarter(date))           AS year_quarter,

  -- Month
  month(date)                                       AS month,
  date_format(date, 'MMMM')                         AS month_name,
  date_format(date, 'MMM')                          AS month_short_name,
  date_format(date, 'yyyy-MM')                      AS year_month,

  -- Week
  weekofyear(date)                                  AS week_of_year,

  -- Day
  day(date)                                         AS day,
  dayofyear(date)                                   AS day_of_year,
  dayofweek(date)                                   AS day_of_week,          -- 1=Sun … 7=Sat
  date_format(date, 'EEEE')                         AS day_name,
  date_format(date, 'EEE')                          AS day_short_name,

  -- Flags
  CASE WHEN dayofweek(date) IN (1, 7) THEN TRUE ELSE FALSE END AS is_weekend,
  CASE WHEN dayofweek(date) NOT IN (1, 7) THEN TRUE ELSE FALSE END AS is_weekday,

  -- Period boundaries
  trunc(date, 'MM')                                 AS first_day_of_month,
  last_day(date)                                    AS last_day_of_month,
  day(last_day(date))                               AS days_in_month,
  trunc(date, 'quarter')                            AS first_day_of_quarter,
  date_add(add_months(trunc(date, 'quarter'), 3), -1) AS last_day_of_quarter,
  trunc(date, 'year')                               AS first_day_of_year,
  DATE(CONCAT(year(date), '-12-31'))                 AS last_day_of_year
FROM date_range
ORDER BY date;

In [0]:
%skip
SELECT count(*) AS total_rows,
       min(date) AS min_date,
       max(date) AS max_date
FROM   adb_qnxthealth.silver.calendar;
